In [1]:
from pynq import Overlay, allocate
import numpy as np
import matplotlib.pyplot as plt

# Load the overlay using your specific filenames
ol = Overlay("dac_adc.bit")

# Check if the blocks are detected (names should match your Vivado design)
print("IP Blocks found:", ol.ip_dict.keys())

# Create handles for the DMA and RFDC
dma = ol.axi_dma_0
rfdc = ol.usp_rf_data_converter_0

IP Blocks found: dict_keys(['axi_dma_0', 'axi_intc_0', 'usp_rf_data_converter_0', 'zynq_ultra_ps_e_0'])


In [36]:
import numpy as np
dma_send = ol.axi_dma_0.sendchannel
data_size = 1000
input_buffer = allocate(shape=(data_size,), dtype=np.uint32)

# Square wave parameters (in hex)
high_val = 0x7FFF   # Max 14-bit value
lev3_val = 0x5555
lev2_val = 0x2AAB
low_val  = 0x000   # Min value
period   = 8       # Number of samples per full wave cycle

for i in range(data_size):
#    if (i % period) < (period // 4):
#        input_buffer[i] = high_val
#    elif (i % period) < (period // 2):
#        input_buffer[i] = lev3_val
#    elif (i % period) < (3 * aperiod // 4):
#        input_buffer[i] = lev2_val
#    else:
#        input_buffer[i] = low_val
# Test: 16-bit Signed Square Wave
# High = 20000, Low = -20000
    if (i % 8) < (8//2):
        input_buffer[i] = 20000
    else:
        input_buffer[i] = -20000
input_buffer.flush()
dma_send.transfer(input_buffer)
#    input_buffer[i] = 2**15-1


# Print first 10 values in hex
for i in range(10):
    print(hex(input_buffer[i]))
try:
    while True:
        dma_send.transfer(input_buffer)
        dma_send.wait()
except KeyboardInterrupt:
    print("Transfer stopped by user.")

0x4e20
0x4e20
0x4e20
0x4e20
0xffffb1e0
0xffffb1e0
0xffffb1e0
0xffffb1e0
0x4e20
0x4e20


RuntimeError: DMA channel not started

In [12]:
import numpy as np

dma_send = ol.axi_dma_0.sendchannel
data_size = 1000 # Ideally a multiple of the period for a smooth loop
input_buffer = allocate(shape=(data_size,), dtype=np.uint32)

# Sine wave parameters
amplitude = 0x3FFF  # Half of max range (for 14-bit: 16383 / 2)
offset = 0x3FFF     # Center point (so wave stays positive)
period = 32# Number of samples per full sine cycle

for i in range(data_size):
    # Calculate sine value: sin(2 * pi * i / period)
    # This generates a value between -1 and 1
    sine_val = np.sin(2 * np.pi * i / period)
    
    # Scale and shift to integer range
    # Resulting range: 0 to 0x7FFE (approx 32766)
    input_buffer[i] = int(amplitude * sine_val + offset)

# Print first 10 values in hex to verify
for i in range(10):
    print(f"Index {i}: {hex(input_buffer[i])}")

try:
    print("Starting continuous sine wave transfer...")
    while True:
        dma_send.transfer(input_buffer)
        dma_send.wait()
except KeyboardInterrupt:
    print("Transfer stopped by user.")

Index 0: 0x3fff
Index 1: 0x4c7b
Index 2: 0x587c
Index 3: 0x638c
Index 4: 0x6d3f
Index 5: 0x7534
Index 6: 0x7b1e
Index 7: 0x7ec3
Index 8: 0x7ffe
Index 9: 0x7ec3
Starting continuous sine wave transfer...
Transfer stopped by user.


In [21]:
import numpy as np

# --- Configuration ---
fs = 307e6            # Sampling frequency (e.g., 100 MHz)
f_target = 1e6        # Desired output frequency (e.g., 1 MHz)
data_size = 1024      # Total number of samples in buffer
amplitude = 0x00FF    # 14-bit amplitude
offset = 0x3FFF       # Center the wave

# --- Buffer Allocation ---
input_buffer = allocate(shape=(data_size,), dtype=np.uint32)

# --- Vectorized Generation (Much Faster) ---
# Create an array of sample indices [0, 1, 2, ..., data_size-1]
i = np.arange(data_size)

# Apply the sine formula
# y = A * sin(2 * pi * f_target * t) where t = i / fs
sine_wave = amplitude * np.sin(2 * np.pi * f_target * i / fs) + offset

# Fill the DMA buffer
input_buffer[:] = sine_wave.astype(np.uint32)

# --- DMA Transfer ---
try:
    print(f"Generating {f_target/1e6} MHz signal...")
    while True:
        dma_send.transfer(input_buffer)
        dma_send.wait()
except KeyboardInterrupt:
    print("Stopped.")

Generating 1.0 MHz signal...
Stopped.


In [33]:
import numpy as np

# --- Parameters from your screenshot ---
axi_stream_clk = 307.2e6
samples_per_cycle = 8
fs = axi_stream_clk * samples_per_cycle  # 2457.6 MHz
f_target = 30e6                          # Desired Frequency (10 MHz)

# --- Buffer Configuration ---
# To avoid the 'jump' glitch, data_size should be a multiple of 
# (fs / f_target) and also a multiple of 8.
# For 10MHz @ 2457.6fs, one cycle is 245.76 samples.
# Let's use a large buffer to minimize rounding errors.
data_size = 30720  
input_buffer = allocate(shape=(data_size,), dtype=np.int16)

# --- Sine Generation ---
amplitude = 32000  # Max for 16-bit is 32767
t = np.arange(data_size)
sine_wave = amplitude * np.sin(2 * np.pi * f_target * t / fs)

# Load into DMA buffer
input_buffer[:] = sine_wave.astype(np.int16)

print(f"Sampling Frequency: {fs/1e6} MHz")
print(f"Target Frequency: {f_target/1e6} MHz")

try:
    while True:
        dma_send.transfer(input_buffer)
        dma_send.wait()
except KeyboardInterrupt:
    print("Stopped.")

Sampling Frequency: 2457.6 MHz
Target Frequency: 30.0 MHz
Stopped.


In [30]:
import numpy as np

# ---------------- Configuration ----------------
fs_dac_in = 2.4576e9        # DAC input sample rate (Hz)
f_out = 1e3                 # Desired analog output frequency (1 MHz)
data_size = 1024            # DMA buffer size

amplitude = 0x3FFF          # 14-bit DAC amplitude
offset = 0x3FFF             # Mid-scale offset

# Allocate DMA buffer (PYNQ / RFSoC)
input_buffer = allocate(shape=(data_size,), dtype=np.uint32)

# Phase accumulator parameters
phase = 0.0
phase_inc = 2 * np.pi * f_out / fs_dac_in

print("Generating 1 MHz sine wave at RFDC output...")

try:
    while True:
        for i in range(data_size):
            input_buffer[i] = int(
                amplitude * np.sin(phase) + offset
            )

            phase += phase_inc
            if phase >= 2 * np.pi:
                phase -= 2 * np.pi

        dma_send.transfer(input_buffer)
        dma_send.wait()

except KeyboardInterrupt:
    print("Stopped.")

Generating 1 MHz sine wave at RFDC output...
Stopped.


In [31]:
import numpy as np

# fs = 307.2 MHz * 8 = 2457.6 MHz
fs = 2457600000.0 
f_target = 10000000.0 # 10 MHz

# data_size must be a multiple of 8 (samples per cycle)
# and for a clean loop, a multiple of (fs / f_target)
data_size = 12288 

# 1. Use np.int16 (Signed)
input_buffer = allocate(shape=(data_size,), dtype=np.int16)

# 2. Generate wave centered at 0 (No Offset)
t = np.arange(data_size)
# Use a slightly lower amplitude to avoid clipping (max is 32767)
amplitude = 28000 
sine_wave = amplitude * np.sin(2 * np.pi * f_target * t / fs)

# 3. Load and Flush
input_buffer[:] = sine_wave.astype(np.int16)
input_buffer.flush() # CRITICAL: Ensures data leaves CPU cache for DMA

dma_send.transfer(input_buffer)
dma_send.wait()

In [4]:
# 1. Allocate buffers (Multiple of 8 samples for 128-bit alignment)
num_samples = 1024 * 16 
send_buffer = allocate(shape=(num_samples,), dtype=np.int16)
recv_buffer = allocate(shape=(num_samples,), dtype=np.int16)

# 2. Generate a test signal (A simple Ramp or Sawtooth is best for testing alignment)
send_buffer[:] = np.arange(num_samples, dtype=np.int16) % 1024

# 3. Perform the Loopback
# We use the names found in your dict_keys
dma = ol.axi_dma_0
dma.sendchannel.start()
dma.recvchannel.start()
dma.recvchannel.transfer(recv_buffer)
dma.sendchannel.transfer(send_buffer)

# Wait for the Concat-linked interrupts to trigger
dma.sendchannel.wait()
dma.recvchannel.wait()

print("Loopback Transfer Successful!")

KeyboardInterrupt: 

In [8]:
import xrfdc

# Manually initialize the driver using the address from your Address Editor
# Note: PYNQ's RFDC driver usually wants the IP dictionary entry
rfdc = xrfdc.RFdc(ol.ip_dict['usp_rf_data_converter_0'])

# If that still fails, we can check the status via direct MMIO as a last resort:
# print(hex(ol.usp_rf_data_converter_0.read(0x228))) # Tile 2 ADC State offset

In [14]:
# Tile 2 ADC State is at offset 0x228 (Tile 2 base) + 0x08 (State offset)
# DAC Tile 0 State is at offset 0x028 (Tile 0 base) + 0x08 (State offset)

adc_state_raw = ol.usp_rf_data_converter_0.read(0x228 + 0x04)
dac_state_raw = ol.usp_rf_data_converter_0.read(0x028 + 0x08)

print(f"ADC Tile 2 State (Raw): {adc_state_raw}")
print(f"DAC Tile 0 State (Raw): {dac_state_raw}")

ADC Tile 2 State (Raw): 0
DAC Tile 0 State (Raw): 33554944


In [4]:
#square wave code


import numpy as np

data_size = 1000
input_buffer = allocate(shape=(data_size), dtype=np.uint16)

# Square wave parameters (in hex)
high_val = 0xFFFE   # Max 14-bit value
lev3_val = 0x5555
lev2_val = 0x2AAB
low_val  = 0x000   # Min value
period   = 8       # Number of samples per full wave cycle

for i in range(data_size):
#    if (i % period) < (period // 4):
#        input_buffer[i] = high_val
#    elif (i % period) < (period // 2):
#        input_buffer[i] = lev3_val
#    elif (i % period) < (3 * period // 4):
#        input_buffer[i] = lev2_val
#    else:
#        input_buffer[i] = low_val
     if (i % period) < (period // 2):
         input_buffer[i] = high_val
     else:
         input_buffer[i] = low_val
#    input_buffer[i] = 2**15-1


# Print first 10 values in hex
for i in range(10):
    print(hex(input_buffer[i]))
try:
    while True:
        dma.sendchannel.transfer(input_buffer)
        dma.sendchannel.wait()
except KeyboardInterrupt:
    print("Transfer stopped by user.")

0xfffe
0xfffe
0xfffe
0xfffe
0x0
0x0
0x0
0x0
0xfffe
0xfffe
Transfer stopped by user.
